# AI-Related Policy Activity Across Indian States — Research Pipeline

**Goal:** collect, screen, classify and export news about **AI-related policy activity by Indian state governments**. The notebook produces (1) a candidate dataset for likely policy activity and (2) a review queue that preserves AI-related stories needing human judgement.

The required core fields are:

`state`, `headline`, `date`, `source_name`, `source_url`, `summary`, `activity_type`

The collection is configured for all 28 Indian states. Eight of those states also have direct regional publisher feeds; all 28 use state-specific Google News RSS discovery queries. This broadens geographic discovery, but does not guarantee uniform source coverage or complete coverage of every announcement.

## What counts as AI-related policy activity?

Include candidate records when the story connects AI / artificial intelligence to a concrete state-government action, for example:

1. a state AI policy, mission, strategy or roadmap;
2. an AI task force, centre of excellence, institute or government-backed lab;
3. AI deployment in public services such as health, agriculture, policing, education or administration;
4. a state-government MoU or partnership with a company, university or research institution for AI;
5. an AI-specific public budget allocation, grant, procurement or investment;
6. state AI regulation, ethics, safety or governance measures; or
7. government-backed AI compute, data-centre or other enabling infrastructure.

Do **not** treat a private company's AI product launch, a general technology event, a generic digital-government announcement, or an AI story with no evidenced state-government connection as confirmed policy activity. Borderline stories go to the review queue. A news report is evidence of a reported announcement, not proof that a policy was implemented.

## Screening policy and evidence limits

- Discovery uses state-specific Google News RSS queries focused on AI policy themes, plus regional publisher feeds as a supplementary discovery source.
- A deterministic keyword screen checks for an AI signal, a concrete activity signal, and an action linguistically connected to a government/public-sector actor. A government mention alone is insufficient.
- Records matching all three are exported as **Include candidate — verify manually**. Partial matches are labelled **Needs manual review**. AI stories without a policy signal are retained as **Exclude — no clear policy activity** in the review/audit file, not the candidate dataset.
- The state is initially assigned from the configured state-specific feed/query. A state/city mention in the headline or feed summary strengthens the evidence, but does not replace manual checking.
- The notebook reads RSS headlines and short feed summaries; it does not establish facts from full article bodies. It does not use an LLM to generate facts or summaries.

> RSS availability, feed ordering and Google News results vary over time. This is a reproducible collection procedure, not a complete census. Check publishers' terms before reuse or redistribution, preserve attribution and link to the original story.

## 1. Source catalogue and useful news links

These are collection inputs and places to verify announcements. The notebook builds a state-specific Google News RSS query catalogue for AI themes and supplements it with regional publisher feeds.

### Source directories

- [The Indian Express — official RSS feed directory](https://indianexpress.com/rss/)
- [The Hindu — RSS feed directory](https://www.thehindu.com/rssfeeds/)
- [The New Indian Express — States page](https://www.newindianexpress.com/states)
- [Google News India](https://news.google.com/?hl=en-IN&gl=IN&ceid=IN:en)

### Practical interpretation examples

| Story type | Screening treatment | Rationale |
|---|---|---|
| State announces an AI mission or policy | Include candidate | Explicit state AI policy action |
| Government launches AI-based crop or health service | Include candidate | AI deployment in a public service |
| State signs an AI research MoU with a university | Include candidate | Government-linked AI partnership |
| State budget provides a specific AI allocation | Include candidate | Public funding for AI |
| Private startup launches an AI app without a state-government role | Exclude | Commercial AI news is not state policy activity |
| General digital mission with no explicit AI component | Exclude | AI connection is not established |
| Article mentions AI and a state, but no concrete government action | Needs manual review | The article may be context rather than a policy announcement |

**Coverage rationale:** the 28-state pilot provides a manageable first pass across all five major regions of India. Results must not be described as a ranking of all Indian states. States with no records may reflect feed/query coverage gaps rather than an absence of activity.

In [ ]:
# Install the lightweight dependencies in the Colab runtime.
%pip -q install requests feedparser beautifulsoup4 pandas python-dateutil rapidfuzz openpyxl tqdm

In [ ]:
from __future__ import annotations

import calendar
import html
import json
import re
import time
import hashlib
from collections import defaultdict
from datetime import datetime, timezone, timedelta
from email.utils import parsedate_to_datetime
from urllib.parse import quote_plus, urlsplit, urlunsplit, parse_qsl, urlencode

import feedparser
import pandas as pd
import requests
from bs4 import BeautifulSoup
from dateutil import parser as date_parser
from rapidfuzz import fuzz
from tqdm.auto import tqdm
from IPython.display import display, HTML

pd.set_option("display.max_colwidth", 140)

USER_AGENT = ("Mozilla/5.0 (compatible; AIStatePolicyTracker/1.0;+https://colab.research.google.com/)")
SESSION = requests.Session()
SESSION.headers.update({"User-Agent": USER_AGENT, "Accept": "application/rss+xml, application/xml, text/xml, */*"})


## 2. Configure the AI-policy pilot

Edit the states, lookback window, feed limits and screening thresholds below. The discovery queries are intentionally AI-specific. Regional feeds are supplementary for eight states; all 28 states receive four targeted Google News RSS query groups. Regional/general-news records are screened out unless they contain an AI signal.

`DAYS_BACK` defaults to 365 because state AI-policy announcements are less frequent than general daily news. Increase the window only when you can still review the resulting records and confirm source dates.

In [ ]:
# Coverage: all 28 Indian states. Regional publisher feeds supplement Google News
# discovery for the eight states with configured direct feeds below.
STATES = [
    "Andhra Pradesh", "Arunachal Pradesh", "Assam", "Bihar", "Chhattisgarh",
    "Goa", "Gujarat", "Haryana", "Himachal Pradesh", "Jharkhand",
    "Karnataka", "Kerala", "Madhya Pradesh", "Maharashtra", "Manipur",
    "Meghalaya", "Mizoram", "Nagaland", "Odisha", "Punjab", "Rajasthan",
    "Sikkim", "Tamil Nadu", "Telangana", "Tripura", "Uttar Pradesh",
    "Uttarakhand", "West Bengal",]


In [ ]:

MAX_ITEMS_PER_FEED = 20
REQUEST_TIMEOUT = 15
REQUEST_PAUSE_SECONDS = 0.25
DAYS_BACK = 365
DEDUP_SIMILARITY_THRESHOLD = 91
DEDUP_DATE_WINDOW_DAYS = 4

STATE_ALIASES = {
    "andhra pradesh": "Andhra Pradesh", "andhra-pradesh": "Andhra Pradesh", "ap": "Andhra Pradesh",
    "telangana": "Telangana", "tg": "Telangana",
    "karnataka": "Karnataka", "ka": "Karnataka",
    "tamil nadu": "Tamil Nadu", "tamilnadu": "Tamil Nadu", "tn": "Tamil Nadu",
    "kerala": "Kerala", "keralam": "Kerala",
    "maharashtra": "Maharashtra", "mh": "Maharashtra",
    "uttar pradesh": "Uttar Pradesh", "uttar-pradesh": "Uttar Pradesh", "up": "Uttar Pradesh",
    "rajasthan": "Rajasthan", "rj": "Rajasthan",
    "arunachal pradesh": "Arunachal Pradesh", "assam": "Assam", "bihar": "Bihar",
    "chhattisgarh": "Chhattisgarh", "goa": "Goa", "gujarat": "Gujarat",
    "haryana": "Haryana", "himachal pradesh": "Himachal Pradesh", "jharkhand": "Jharkhand",
    "madhya pradesh": "Madhya Pradesh", "manipur": "Manipur", "meghalaya": "Meghalaya",
    "mizoram": "Mizoram", "nagaland": "Nagaland", "odisha": "Odisha", "orissa": "Odisha",
    "punjab": "Punjab", "sikkim": "Sikkim", "tripura": "Tripura",
    "uttarakhand": "Uttarakhand", "uttaranchal": "Uttarakhand", "west bengal": "West Bengal",}

# Regional publisher feeds are broad discovery supplements, not proof that an
# individual story is an AI-policy story. The AI relevance screen runs later.
DIRECT_FEEDS = {
    "Andhra Pradesh": [
        ("The Hindu — Andhra Pradesh", "https://www.thehindu.com/news/national/andhra-pradesh/feeder/default.rss"),
    ],
    "Telangana": [
        ("The Hindu — Telangana", "https://www.thehindu.com/news/national/telangana/feeder/default.rss"),
        ("The Indian Express — Hyderabad", "https://indianexpress.com/section/cities/hyderabad/feed/"),
    ],
    "Karnataka": [
        ("The Hindu — Karnataka", "https://www.thehindu.com/news/national/karnataka/feeder/default.rss"),
        ("The Indian Express — Bengaluru", "https://indianexpress.com/section/cities/bangalore/feed/"),
    ],
    "Tamil Nadu": [
        ("The Hindu — Tamil Nadu", "https://www.thehindu.com/news/national/tamil-nadu/feeder/default.rss"),
        ("The Indian Express — Chennai", "https://indianexpress.com/section/cities/chennai/feed/"),
    ],
    "Kerala": [
        ("The Hindu — Kerala", "https://www.thehindu.com/news/national/kerala/feeder/default.rss"),
        ("The Indian Express — Kerala", "https://indianexpress.com/section/india/kerala/feed/"),
    ],
    "Maharashtra": [
        ("The Indian Express — Mumbai", "https://indianexpress.com/section/cities/mumbai/feed/"),
        ("The Indian Express — Pune", "https://indianexpress.com/section/cities/pune/feed/"),
    ],
    "Uttar Pradesh": [
        ("The Indian Express — Lucknow", "https://indianexpress.com/section/cities/lucknow/feed/"),
    ],
    "Rajasthan": [
        ("The Indian Express — Jaipur", "https://indianexpress.com/section/cities/jaipur/feed/"),
    ],
}

# Four complementary discovery queries per state. Google News RSS is a discovery
# endpoint rather than a documented service-level API. These queries intentionally
# include variations in policy, institutions, deployment, partnerships and funding.


In [ ]:
AI_POLICY_QUERY_GROUPS = {
    "Policies and institutions": '"AI policy" OR "AI mission" OR "AI strategy" OR "AI roadmap" OR "AI task force" OR "AI centre of excellence" OR "AI center of excellence"',
    "Government deployments": '"artificial intelligence" government OR "AI" department OR "AI-powered public service" OR "AI" agriculture OR "AI" healthcare OR "AI" policing OR "AI" education',
    "Partnerships and funding": '"AI" MoU OR "AI" partnership OR "AI" budget OR "AI" funding OR "AI" investment OR "AI" procurement',
    "Regulation and infrastructure": '"AI" regulation OR "AI" governance OR "responsible AI" OR "AI data centre" OR "AI data center" OR "AI compute" OR "AI infrastructure"',
}

GOOGLE_QUERY_TEMPLATE = '"{state}" ({topic_query}) when:{days}d'

def google_news_rss_url(state: str, topic_query: str) -> str:
    query = quote_plus(GOOGLE_QUERY_TEMPLATE.format(state=state, topic_query=topic_query, days=DAYS_BACK))
    return f"https://news.google.com/rss/search?q={query}&hl=en-IN&gl=IN&ceid=IN:en"

source_rows = []
for state in STATES:
    for l, url in DIRECT_FEEDS.get(state, []):
        source_rows.append({
            "state": state, "source_l": l,
            "feed_type": "Publisher RSS — regional discovery", "url": url,
        })
    for group_name, topic_query in AI_POLICY_QUERY_GROUPS.items():
        source_rows.append({
            "state": state,
            "source_l": f"Google News — {state} — {group_name}",
            "feed_type": "Google News RSS — AI-policy discovery",
            "url": google_news_rss_url(state, topic_query),
        })

sc = pd.DataFrame(source_rows)
display(HTML(sc.to_html(escape=False, render_links=True, index=False)))
print(f"Configured feeds: {len(sc)} across {len(STATES)} states")

## 3. Fetch and parse RSS feeds

This cell retrieves RSS XML with `requests`, then parses it with `feedparser`. A failed feed is logged and skipped rather than stopping the entire notebook. No login, paywall bypass or browser automation is attempted.


In [ ]:
def fetch_feed(url: str):
    """Fetch one RSS/Atom feed. Raises a useful exception on HTTP/network failure."""
    response = SESSION.get(url, timeout=REQUEST_TIMEOUT)
    response.raise_for_status()
    parsed = feedparser.parse(response.content)
    if parsed.bozo and not parsed.entries:
        raise ValueError(f"Feed parse error: {parsed.bozo_exception}")
    return parsed


def parse_entry_datetime(entry) -> tuple[pd.Timestamp | pd.NaT, str]:
    """Return UTC timestamp and a transparent date-status l."""
    for field, status in (("published_parsed", "parsed_published"), ("updated_parsed", "parsed_updated")):
        value = entry.get(field)
        if value:
            try:
                ts = datetime.fromtimestamp(calendar.timegm(value), tz=timezone.utc)
                return pd.Timestamp(ts), status
            except Exception:
                pass
    for field, status in (("published", "parsed_published_text"), ("updated", "parsed_updated_text"), ("date", "parsed_date_text")):
        raw = entry.get(field)
        if raw:
            try:
                dt = date_parser.parse(raw)
                if dt.tzinfo is None:
                    dt = dt.replace(tzinfo=timezone.utc)
                return pd.Timestamp(dt.astimezone(timezone.utc)), status
            except Exception:
                continue
    return pd.NaT, "missing_or_invalid"


def strip_html(raw_text: str) -> str:
    if not raw_text:
        return ""
    soup = BeautifulSoup(raw_text, "html.parser")
    text = soup.get_text(" ", strip=True)
    text = html.unescape(text)
    text = re.sub(r"\s+", " ", text).strip()
    return text


def normalize_url(url: str) -> str:
    """Remove fragments and common tracking parameters for exact URL matching."""
    if not url:
        return ""
    try:
        parts = urlsplit(url.strip())
        tracking = {"utm_source", "utm_medium", "utm_campaign", "utm_term", "utm_content", "ocid", "fbclid", "gclid"}
        query = [(k, v) for k, v in parse_qsl(parts.query, keep_blank_values=True) if k.lower() not in tracking]
        netloc = parts.netloc.lower()
        path = parts.path.rstrip("/") or "/"
        return urlunsplit((parts.scheme.lower(), netloc, path, urlencode(query, doseq=True), ""))
    except Exception:
        return url.strip()


def source_name_from_entry(entry, fallback_l: str) -> str:
    source = entry.get("source") or {}
    name = source.get("title") if hasattr(source, "get") else None
    if name:
        return strip_html(str(name))
    # Google News search feed sometimes exposes the publisher under `source`;
    # use the configured publisher/feed l as a safe fallback.
    return fallback_l.split(" — ")[0]


def entry_to_record(entry, state: str, feed_l: str, url: str, feed_type: str = "") -> dict:
    headline = strip_html(entry.get("title", ""))
    summary_raw = entry.get("summary", "")
    if not summary_raw and entry.get("content"):
        summary_raw = entry["content"][0].get("value", "")
    summary = strip_html(summary_raw)
    if not summary:
        summary = headline  # explicit fallback; no LLM-generated summary
    # Keep the summary genuinely brief and one-line.
    summary = re.split(r"(?<=[.!?])\s+", summary)[0][:420].strip()

    published_at, date_status = parse_entry_datetime(entry)
    url = normalize_url(entry.get("link", ""))
    if not headline or not url:
        return {}

    source_name = source_name_from_entry(entry, feed_l)
    return {
        "state": STATE_ALIASES.get(state.lower(), state),
        "headline": headline,
        "date": published_at,
        "source_name": source_name,
        "source_url": url,
        "summary": summary,
        "category": "Pending AI-policy screening",
        "date_status": date_status,
        "collection_feed": url,
        "collection_feed_l": feed_l,
        "feed_type": feed_type,
    }


raw_records = []
feed_log = []

for row in tqdm(sc.to_dict("records"), desc="Fetching RSS feeds"):
    state = row["state"]
    l = row["source_l"]
    url = row["url"]
    feed_type = row.get("feed_type", "")
    try:
        parsed = fetch_feed(url)
        entries = parsed.entries[:MAX_ITEMS_PER_FEED]
        added = 0
        for entry in entries:
            record = entry_to_record(entry, state, l, url, feed_type)
            if record:
                raw_records.append(record)
                added += 1
        feed_log.append({"state": state, "source_l": l, "status": "ok", "entries_seen": len(entries), "records_added": added, "error": ""})
    except Exception as exc:
        feed_log.append({"state": state, "source_l": l, "status": "failed", "entries_seen": 0, "records_added": 0, "error": str(exc)[:300]})
    time.sleep(REQUEST_PAUSE_SECONDS)

feed_log_df = pd.DataFrame(feed_log)
raw_df = pd.DataFrame(raw_records)
print(f"Raw records collected: {len(raw_df):,}")
display(feed_log_df)
if raw_df.empty:
    raise RuntimeError("No records were collected. Check the feed log, internet access and source URLs before proceeding.")

## 4. Normalize and validate dates, state names and fields

The state label is inherited from the configured state-specific feed/query. This is a discovery assignment, not conclusive proof of the story's geography. Later screening records whether the headline/summary explicitly names the state or a major city associated with it.

Missing or ambiguous publication dates remain null; the notebook never substitutes today's date. Records with valid dates are restricted to the configured lookback window. Undated records are retained and flagged.

In [ ]:
# Normalize text and canonical state ls.
raw_df["state"] = raw_df["state"].fillna("").astype(str).str.strip().map(
    lambda x: STATE_ALIASES.get(x.lower(), x)
)
raw_df["headline"] = raw_df["headline"].fillna("").astype(str).str.replace(r"\s+", " ", regex=True).str.strip()
raw_df["summary"] = raw_df["summary"].fillna("").astype(str).str.replace(r"\s+", " ", regex=True).str.strip()
raw_df["source_name"] = raw_df["source_name"].fillna("Unknown source").astype(str).str.strip()
raw_df["source_url"] = raw_df["source_url"].fillna("").map(normalize_url)
raw_df["date"] = pd.to_datetime(raw_df["date"], utc=True, errors="coerce")

# Optional freshness filter: preserve undated records, but flag them separately.
cutoff = pd.Timestamp.now(tz="UTC") - pd.Timedelta(days=DAYS_BACK)
valid_date_mask = raw_df["date"].isna() | (raw_df["date"] >= cutoff)
working_df = raw_df.loc[valid_date_mask].copy()
if working_df.empty:
    raise RuntimeError("No records remain after the lookback filter. Increase DAYS_BACK or inspect the feed log and source dates.")

# Exact duplicates by canonical article URL (before fuzzy title matching).
working_df = working_df.drop_duplicates(subset=["state", "source_url"], keep="first").reset_index(drop=True)

required_nonempty = ["state", "headline", "source_name", "source_url"]
for col in required_nonempty:
    working_df[col] = working_df[col].astype(str).str.strip()

working_df["record_valid"] = working_df[required_nonempty].apply(
    lambda row: all(bool(value) and value.lower() != "nan" for value in row), axis=1
)
working_df = working_df[working_df["record_valid"]].copy()

print("Records after freshness filter and exact-URL deduplication:", len(working_df))
print("Records with missing/invalid dates:", int(working_df["date"].isna().sum()))
display(working_df[["state", "headline", "date", "source_name", "category", "date_status"]].head(10))


## 5. Near-duplicate detection (headline similarity)

The goal is to collapse the same announcement reported by several outlets, not stories that are merely about the same AI topic. Records are compared only within the same state, with a high headline similarity threshold and a four-day date window when both dates exist. A direct publisher URL is preferred over a Google News redirect, and alternate sources are retained for audit.

In [ ]:
def normalize_headline(text: str) -> str:
    text = (text or "").lower()
    text = re.sub(r"https?://\S+", " ", text)
    text = re.sub(r"[^a-z0-9]+", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text


def is_google_redirect(url: str) -> bool:
    host = urlsplit(url).netloc.lower()
    return host.endswith("google.com") or host.endswith("google.co.in") or host.endswith("news.google.com")


def dates_compatible(a, b, window_days: int = DEDUP_DATE_WINDOW_DAYS) -> bool:
    if pd.isna(a) or pd.isna(b):
        return True  # no fabricated date; let headline similarity decide
    return abs((a - b).total_seconds()) <= window_days * 86400


def deduplicate_headlines(df: pd.DataFrame, threshold: int = DEDUP_SIMILARITY_THRESHOLD) -> pd.DataFrame:
    records = df.to_dict("records")
    clusters = []

    # Compare within the same state only. Pilot volume is intentionally small.
    for rec in tqdm(records, desc="Finding likely duplicate headlines"):
        title = normalize_headline(rec["headline"])
        best_cluster_idx = None
        best_score = -1
        for idx, cluster in enumerate(clusters):
            representative = cluster[0]
            if rec["state"] != representative["state"]:
                continue
            if not dates_compatible(rec["date"], representative["date"]):
                continue
            score = fuzz.token_set_ratio(title, normalize_headline(representative["headline"]))
            if score >= threshold and score > best_score:
                best_score = score
                best_cluster_idx = idx
        if best_cluster_idx is None:
            clusters.append([rec])
        else:
            clusters[best_cluster_idx].append(rec)

    deduped = []
    for cluster in clusters:
        # Prefer original publisher article URLs over Google News redirect URLs.
        def representative_score(item):
            direct_bonus = 1000 if not is_google_redirect(item["source_url"]) else 0
            summary_score = min(len(item.get("summary", "")), 500)
            dated_bonus = 20 if pd.notna(item.get("date")) else 0
            return direct_bonus + summary_score + dated_bonus

        representative = max(cluster, key=representative_score).copy()
        all_urls = list(dict.fromkeys(x["source_url"] for x in cluster if x.get("source_url")))
        all_sources = list(dict.fromkeys(x["source_name"] for x in cluster if x.get("source_name")))
        representative["duplicate_count"] = len(cluster)
        representative["also_reported_by"] = " | ".join(all_sources)
        representative["alternate_source_urls"] = " | ".join(
            url for url in all_urls if url != representative["source_url"]
        )
        deduped.append(representative)

    result = pd.DataFrame(deduped)
    if not result.empty:
        result = result.sort_values(["state", "date"], ascending=[True, False], na_position="last").reset_index(drop=True)
    return result


dedup_df = deduplicate_headlines(working_df)
print(f"Before fuzzy deduplication: {len(working_df):,}")
print(f"After fuzzy deduplication:  {len(dedup_df):,}")
print(f"Likely duplicate records collapsed: {len(working_df) - len(dedup_df):,}")
display(dedup_df[["state", "headline", "source_name", "duplicate_count", "also_reported_by"]].head(15))


## 6. AI-policy relevance screening and activity classification

This is the central assessment-specific step. The screen uses transparent, editable keyword patterns rather than an opaque model. It evaluates AI evidence, concrete activity evidence, and whether an action is linguistically connected to a government/public-sector actor. A bare mention of government is not sufficient.

**Interpret the outcome carefully:** `Include candidate — verify manually` is a prioritised candidate, not a claim that a human has verified the announcement. `Needs manual review` preserves incomplete or ambiguous matches. AI product/company news with no clear government policy signal is marked for exclusion from the candidate dataset but remains in the review/audit export.

In [ ]:
# Transparent rule-based screening. Edit these ps if review reveals a missed
# synonym or a false positive. Matching uses headline + short RSS summary only.
AI_pS = {
    "artificial intelligence": r"\bartificial intelligence\b",
    "generative AI": r"\bgenerative\s+ai\b",
    "machine learning": r"\bmachine\s+learning\b",
    "deep learning": r"\bdeep\s+learning\b",
    "AI acronym": r"\bAI\b",
    "AI-powered / AI-based": r"\bAI[- ](?:powered|based|enabled|driven)\b",
}

GOVERNMENT_pS = {
    "state government": r"\bstate government\b|\bgovernment of [a-z ]+",
    "government / govt": r"\bgovernment\b|\bgovt\b",
    "chief minister / CM": r"\bchief minister\b|\bCM\b",
    "minister / cabinet": r"\bminister\b|\bcabinet\b",
    "government department": r"\bdepartment\b|\bdirectorate\b|\bsecretariat\b",
    "public administration": r"\bpublic sector\b|\bstate[- ]run\b|\bstate[- ]led\b|\bmunicipal corporation\b|\bdistrict administration\b",
}

# A government mention alone is not enough. For candidate inclusion, require an
# action to be linguistically connected to a government/public actor. This avoids
# treating criticism, demands directed at government, or private AI projects as
# state-government initiatives just because the word government appears.
GOVERNMENT_ACTION_VERBS = (
    r"announc\w*|introduc\w*|unveil\w*|approv\w*|sanction\w*|"
    r"establish\w*|set up|form(?:ed|ing)?|constitut\w*|launch\w*|"
    r"deploy\w*|roll(?:ed)? out|sign(?:ed|s)?|"
    r"allocat\w*|fund(?:ed|ing)?|procure\w*|adopt\w*|notifi\w*|"
    r"issue\w*|creat\w*|start(?:ed|s)?|operationali[sz]e\w*"
)
GOVERNMENT_ACTOR_p = (
    r"state government|government|govt|chief minister|\bCM\b|minister|"
    r"cabinet|department|directorate|secretariat|public sector|municipal corporation|"
    r"district administration"
)
STATE_ACTOR_p = "|".join(re.escape(state) for state in STATES)
GOVERNMENT_ACTION_p = (
    rf"(?:\b(?:{GOVERNMENT_ACTOR_p})\b).{{0,100}}\b(?:{GOVERNMENT_ACTION_VERBS})\b"
    rf"|\b(?:{GOVERNMENT_ACTION_VERBS})\b.{{0,100}}\b(?:{GOVERNMENT_ACTOR_p})\b"
    # State name as grammatical actor, e.g. "Karnataka announces an AI mission".
    rf"|\b(?:{STATE_ACTOR_p})\b.{{0,80}}\b(?:{GOVERNMENT_ACTION_VERBS})\b"
)

ACTION_pS = {
    "announced / introduced": r"\bannounc\w*\b|\bintroduc\w*\b|\bunveil\w*\b|\breleas\w*\b",
    "approved / sanctioned": r"\bapprov\w*\b|\bsanction\w*\b|\bclear(?:ed|ance)\b",
    "established / formed": r"\bestablish\w*\b|\bset up\b|\bform(?:ed|ing)?\b|\bconstitut\w*\b|\blaunch\w*\b",
    "deployed / implemented": r"\bdeploy\w*\b|\bimplement\w*\b|\broll(?:ed)? out\b|\boperational\b|\bpilot(?:ed)?\b",
    "agreement / partnership": r"\bMoU\b|\bmemorandum of understanding\b|\bpartnership\b|\bcollaborat\w*\b|\bagreement\b|\bsign(?:ed|s)?\b",
    "funding / investment": r"\bbudget\b|\ballocat\w*\b|\bfund(?:ing)?\b|\bgrant\b|\binvest(?:ment|ed|s)?\b|\bprocure\w*\b",
    "planned / proposed": r"\bpropos\w*\b|\bplan(?:ned|s|ning)?\b|\broadmap\b|\bintent\b|\bconsider\w*\b",
    "regulation / governance": r"\bregulat\w*\b|\bguideline\w*\b|\bethic\w*\b|\bgovernance\b|\bframework\b|\bpolicy\b|\brule\w*\b|\bnotifi\w*\b",
}

ACTIVITY_CATEGORY_pS = {
    "AI Policy & Strategy": [
        r"\bAI policy\b", r"\bartificial intelligence policy\b", r"\bAI mission\b",
        r"\bAI strateg\w*\b", r"\bAI roadmap\b", r"\bAI vision\b", r"\bAI action plan\b",
    ],
    "Institutional Initiatives": [
        r"\bAI task force\b", r"\b(?:centre|center) of excellence\b", r"\bAI institute\b",
        r"\bAI lab(?:oratory)?\b", r"\bAI research centre\b", r"\bAI research center\b",
    ],
    "Partnerships & MoUs": [
        r"\bMoU\b", r"\bmemorandum of understanding\b", r"\bpartnership\b",
        r"\bcollaborat\w*\b", r"\bconsortium\b", r"\bagreement with\b",
    ],
    "Funding & Budget": [
        r"\bAI budget\b", r"\bbudget allocation\b", r"\bfunding\b", r"\bgrant\b",
        r"\ballocat\w*\b", r"\binvestment\b", r"\bprocurement\b",
    ],
    "AI Regulation & Ethics": [
        r"\bAI regulation\b", r"\bAI governance\b", r"\bresponsible AI\b",
        r"\bAI ethics\b", r"\bAI safety\b", r"\bAI guideline\w*\b", r"\bAI framework\b",
    ],
    "AI Infrastructure": [
        r"\bAI data cent(?:re|er)s?\b", r"\bdata cent(?:re|er)s?\b", r"\bAI compute\b",
        r"\bGPU cluster\b", r"\bAI infrastructure\b", r"\bAI cloud\b", r"\bsupercomput\w*\b",
    ],
    "Government AI Deployment": [
        r"\bAI[- ]powered\b", r"\bAI[- ]based\b", r"\bAI[- ]enabled\b",
        r"\busing AI\b", r"\bAI in (?:health|healthcare|agriculture|farming|polic(?:e|ing)|education|governance|administration)\b",
        r"\bAI (?:voice agent|voice assistant|chatbot|helpline)\b",
    ],
}

# State/city mentions are a supporting signal only. State assignment still comes
# from the configured state-specific feed/query and must be manually confirmed.
STATE_GEO_TERMS = {
    "Andhra Pradesh": ["Andhra Pradesh", "Amaravati", "Visakhapatnam", "Vijayawada", "Tirupati"],
    "Arunachal Pradesh": ["Arunachal Pradesh", "Itanagar", "Naharlagun"],
    "Assam": ["Assam", "Dispur", "Guwahati"],
    "Bihar": ["Bihar", "Patna", "Gaya"],
    "Chhattisgarh": ["Chhattisgarh", "Raipur", "Bilaspur"],
    "Goa": ["Goa", "Panaji", "Margao"],
    "Gujarat": ["Gujarat", "Gandhinagar", "Ahmedabad", "Surat"],
    "Haryana": ["Haryana", "Gurugram", "Gurgaon", "Faridabad", "Panchkula"],
    "Himachal Pradesh": ["Himachal Pradesh", "Shimla", "Dharamshala"],
    "Jharkhand": ["Jharkhand", "Ranchi", "Jamshedpur"],
    "Karnataka": ["Karnataka", "Bengaluru", "Bangalore", "Mysuru", "Mangalore", "Mangaluru"],
    "Kerala": ["Kerala", "Keralam", "Thiruvananthapuram", "Kochi", "Kozhikode"],
    "Madhya Pradesh": ["Madhya Pradesh", "Bhopal", "Indore", "Gwalior"],
    "Maharashtra": ["Maharashtra", "Mumbai", "Pune", "Nagpur"],
    "Manipur": ["Manipur", "Imphal"],
    "Meghalaya": ["Meghalaya", "Shillong"],
    "Mizoram": ["Mizoram", "Aizawl"],
    "Nagaland": ["Nagaland", "Kohima", "Dimapur"],
    "Odisha": ["Odisha", "Orissa", "Bhubaneswar", "Cuttack"],
    "Punjab": ["Punjab", "Ludhiana", "Amritsar", "Jalandhar"],
    "Rajasthan": ["Rajasthan", "Jaipur", "Jodhpur", "Udaipur", "Kota"],
    "Sikkim": ["Sikkim", "Gangtok"],
    "Tamil Nadu": ["Tamil Nadu", "Chennai", "Coimbatore", "Madurai"],
    "Telangana": ["Telangana", "Hyderabad", "Warangal"],
    "Tripura": ["Tripura", "Agartala"],
    "Uttar Pradesh": ["Uttar Pradesh", "Lucknow", "Noida", "Kanpur", "Varanasi", "Ayodhya"],
    "Uttarakhand": ["Uttarakhand", "Uttaranchal", "Dehradun", "Haldwani"],
    "West Bengal": ["West Bengal", "Kolkata", "Siliguri"],
}


def find_p_ls(text: str, p_map: dict) -> list[str]:
    text = text or ""
    return [l for l, p in p_map.items() if re.search(p, text, flags=re.IGNORECASE)]


def find_activity_types(text: str) -> list[str]:
    text = text or ""
    found = []
    for activity_type, ps in ACTIVITY_CATEGORY_pS.items():
        if any(re.search(p, text, flags=re.IGNORECASE) for p in ps):
            found.append(activity_type)
    return found


def infer_policy_stage(text: str) -> str:
    t = (text or "").lower()
    if re.search(r"\b(deployed|implemented|rolled out|operational|in use|launched pilot)\b", t):
        return "Implemented / operational (reported)"
    if re.search(r"\b(approved|sanctioned|cleared)\b", t):
        return "Approved / sanctioned (reported)"
    if re.search(r"\b(mou|memorandum of understanding|agreement)\b.{0,80}\b(signed|inked|entered)\b|\b(signed|inked)\b.{0,80}\b(mou|agreement)\b", t):
        return "Agreement signed (reported)"
    if re.search(r"\b(allocated|allocation|funded|grant|released funds)\b", t):
        return "Funding allocated (reported)"
    if re.search(r"\b(proposed|proposal|planned|plan to|draft|considering|roadmap)\b", t):
        return "Proposed / planned (reported)"
    if re.search(r"\b(announc\w*|unveil\w*|introduc\w*|launch\w*)\b", t):
        return "Announced / introduced (reported)"
    return "Unclear — verify article"


def classify_ai_policy_activity(row: dict) -> dict:
    headline = str(row.get("headline", "") or "")
    summary = str(row.get("summary", "") or "")
    text = f"{headline}. {summary}"

    ai_hits = find_p_ls(text, AI_pS)
    gov_hits = find_p_ls(text, GOVERNMENT_pS)
    action_hits = find_p_ls(text, ACTION_pS)
    activity_types = find_activity_types(text)

    state = str(row.get("state", "") or "")
    geo_terms = STATE_GEO_TERMS.get(state, [state] if state else [])
    geo_hits = [term for term in geo_terms if term and re.search(r"\b" + re.escape(term) + r"\b", text, flags=re.IGNORECASE)]

    ai_signal = bool(ai_hits)
    government_signal = bool(gov_hits)
    government_action_signal = bool(re.search(GOVERNMENT_ACTION_p, text, flags=re.IGNORECASE))
    activity_signal = bool(action_hits or activity_types)

    if ai_signal and government_action_signal and activity_signal:
        decision = "Include candidate — verify manually"
    elif ai_signal and (government_signal or activity_signal or geo_hits):
        decision = "Needs manual review"
    elif ai_signal:
        decision = "Exclude — no clear policy activity"
    else:
        decision = "Not AI-related — screened out"

    if ai_signal and government_action_signal and activity_signal and geo_hits:
        confidence = "Higher screening confidence — still verify"
    elif ai_signal and government_action_signal and activity_signal:
        confidence = "Medium screening confidence — state link needs checking"
    elif ai_signal:
        confidence = "Low / incomplete ev"
    else:
        confidence = "Not applicable"

    primary_type = activity_types[0] if activity_types else "Other / review required"
    ev = " | ".join([
        *(f"AI: {x}" for x in ai_hits),
        *(f"Government: {x}" for x in gov_hits),
        *(f"Activity: {x}" for x in action_hits),
    ])
    if government_action_signal and not gov_hits:
        ev = (ev + " | " if ev else "") + "State name matched as acting subject before an action verb"
    if not ev:
        ev = "No matching ev cues in headline/feed summary"

    if geo_hits:
        state = "State/city named in headline or feed summary: " + ", ".join(geo_hits)
    else:
        state = "Assigned from state-specific feed/query only — verify geography"

    return {
        "ai_signal": ai_signal,
        "government_signal": government_signal,
        "government_action_signal": government_action_signal,
        "activity_signal": activity_signal,
        "screening_decision": decision,
        "screening_confidence": confidence,
        "activity_type": primary_type,
        "activity_types": " | ".join(activity_types) if activity_types else "",
        "policy_stage": infer_policy_stage(text),
        "ai_ev_terms": " | ".join(ai_hits),
        "government_ev_terms": " | ".join(gov_hits),
        "activity_ev_terms": " | ".join(action_hits),
        "state_ev": state,
        "relevance_reason": ev,
        "ev_excerpt": summary if summary else headline,
        "manual_review_status": "Pending human verification" if ai_signal else "Not required — no AI signal",
    }


screening_rows = [classify_ai_policy_activity(row) for row in dedup_df.to_dict("records")]
screened_df = pd.concat(
    [dedup_df.reset_index(drop=True), pd.DataFrame(screening_rows)], axis=1
)

# Candidate dataset: all records with AI + government + concrete activity cues.
# The review queue also preserves partial and excluded AI stories for audit.
pc_df = screened_df[
    screened_df["screening_decision"] == "Include candidate — verify manually"
].copy()
ai_review_df = screened_df[screened_df["ai_signal"]].copy()

print(f"Deduplicated records screened: {len(screened_df):,}")
print(f"AI-related records in review queue: {len(ai_review_df):,}")
print(f"Likely policy candidates (not yet human-verified): {len(pc_df):,}")
display(screened_df["screening_decision"].value_counts(dropna=False).rename_axis("screening_decision").reset_index(name="records"))
display(ai_review_df[["state", "headline", "activity_type", "policy_stage", "screening_decision", "screening_confidence"]].head(30))

In [ ]:
# Standardize dates and build the deliverable schema. Missing dates remain blank.
required_schema = [
    "state", "headline", "date", "source_name", "source_url", "summary", "activity_type"
]
review_fields = [
    "activity_types", "policy_stage", "screening_decision", "screening_confidence",
    "manual_review_status", "ai_signal", "government_signal", "government_action_signal", "activity_signal",
    "ai_ev_terms", "government_ev_terms", "activity_ev_terms",
    "state_ev", "relevance_reason", "ev_excerpt",
]
audit_fields = [
    "date_status", "duplicate_count", "also_reported_by", "alternate_source_urls",
    "collection_feed", "collection_feed_l", "feed_type",
]

# Keep all expected columns even when no candidates are found in a particular run.
for col in required_schema + review_fields + audit_fields:
    if col not in pc_df.columns:
        pc_df[col] = ""
    if col not in ai_review_df.columns:
        ai_review_df[col] = ""

pc_df["date"] = pd.to_datetime(pc_df["date"], utc=True, errors="coerce").dt.strftime("%Y-%m-%d").fillna("")
ai_review_df["date"] = pd.to_datetime(ai_review_df["date"], utc=True, errors="coerce").dt.strftime("%Y-%m-%d").fillna("")

candidate_columns = required_schema + review_fields + audit_fields
final_df = pc_df[candidate_columns].copy()
review_df = ai_review_df[candidate_columns].copy()

# Basic integrity checks. An empty candidate set is valid: it means this run did
# not find records meeting all screening rules; never manufacture placeholder rows.
assert set(required_schema).issubset(final_df.columns)
if not final_df.empty:
    assert final_df["headline"].fillna("").str.len().gt(0).all()
    assert final_df["source_url"].fillna("").str.startswith(("http://", "https://")).all()

quality_metrics = [
    ("Raw RSS records collected", len(raw_df)),
    ("Records after normalization / exact URL deduplication", len(working_df)),
    ("Records after fuzzy deduplication", len(dedup_df)),
    ("AI-related records in review queue", len(review_df)),
    ("Likely AI-policy candidates (human verification pending)", len(final_df)),
    ("States represented in candidate dataset", final_df["state"].nunique() if not final_df.empty else 0),
    ("Candidate records with publication date", int(final_df["date"].ne("").sum())),
    ("Candidate records without publication date", int(final_df["date"].eq("").sum())),
    ("Candidate records with non-empty feed summary", int(final_df["summary"].fillna("").str.len().gt(0).sum())),
    ("Candidate records with a valid HTTP(S) article URL", int(final_df["source_url"].fillna("").str.startswith(("http://", "https://")).sum())),
]
quality_report = pd.DataFrame(quality_metrics, columns=["metric", "value"])

print("AI POLICY PIPELINE QUALITY REPORT")
display(quality_report)
print("\nCandidate records by state")
if final_df.empty:
    print("No records met all screening criteria in this run. Inspect ai_policy_review_queue.csv and feed_execution_log.csv.")
else:
    display(final_df.groupby("state").size().rename("candidate_records").reset_index().sort_values("candidate_records", ascending=False))
print("\nCandidate records by activity type")
if not final_df.empty:
    display(final_df.groupby("activity_type").size().rename("candidate_records").reset_index().sort_values("candidate_records", ascending=False))

In [ ]:
# 7. Visual quality checks: charts describe candidate counts only.
import matplotlib.pyplot as plt

if final_df.empty:
    print("No likely AI-policy candidates to chart in this run. Review the AI-policy review queue and source execution log.")
else:
    state_counts = final_df.groupby("state").size().sort_values(ascending=True)
    plt.figure(figsize=(9, 5))
    state_counts.plot(kind="barh")
    plt.title("Likely AI-policy candidates by state (28-state pilot)")
    plt.xlabel("Candidate records")
    plt.ylabel("State")
    plt.tight_layout()
    plt.show()

    category_counts = final_df.groupby("activity_type").size().sort_values(ascending=False)
    plt.figure(figsize=(10, 5))
    category_counts.plot(kind="bar")
    plt.title("Candidate records by AI-policy activity type")
    plt.xl("Activity type")
    plt.yl("Candidate records")
    plt.xticks(rotation=35, ha="right")
    plt.tight_layout()
    plt.show()

## 8. Export dataset, review queue, source log and rationale

The main CSV contains likely policy candidates only. A separate review queue contains every record with an AI signal and its screening evidence, including stories not promoted to the candidate dataset. The Excel workbook includes data, review queue, source catalogue, feed log and quality report.

By default, files are written to `./state_news_output` (which is `/content/state_news_output` in a standard Colab session). Set `STATE_NEWS_OUTPUT_DIR` to choose another folder.

In [ ]:
import os
from pathlib import Path

OUTPUT_DIR = Path(os.getenv("STATE_NEWS_OUTPUT_DIR", str(Path.cwd() / "state_news_output"))).expanduser().resolve()
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

csv_path = OUTPUT_DIR / "ai_policy_activity_candidates.csv"
xlsx_path = OUTPUT_DIR / "ai_policy_activity_candidates.xlsx"
review_path = OUTPUT_DIR / "ai_policy_review_queue.csv"
all_screened_path = OUTPUT_DIR / "all_screened_state_news.csv"
raw_path = OUTPUT_DIR / "raw_feed_records.json"
feed_log_path = OUTPUT_DIR / "feed_execution_log.csv"
quality_path = OUTPUT_DIR / "data_quality_report.csv"
rationale_path = OUTPUT_DIR / "screening_rationale.md"

final_df.to_csv(csv_path, index=False, encoding="utf-8-sig")
review_df.to_csv(review_path, index=False, encoding="utf-8-sig")
screened_df.to_csv(all_screened_path, index=False, encoding="utf-8-sig")

with pd.ExcelWriter(xlsx_path, engine="openpyxl") as writer:
    final_df.to_excel(writer, sheet_name="AI Policy Candidates", index=False)
    review_df.to_excel(writer, sheet_name="AI Review Queue", index=False)
    sc.to_excel(writer, sheet_name="Source Catalogue", index=False)
    feed_log_df.to_excel(writer, sheet_name="Feed Execution Log", index=False)
    quality_report.to_excel(writer, sheet_name="Quality Report", index=False)

raw_export = raw_df.copy()
if "date" in raw_export:
    raw_export["date"] = raw_export["date"].astype(str)
raw_export.to_json(raw_path, orient="records", indent=2, force_ascii=False)
feed_log_df.to_csv(feed_log_path, index=False)
quality_report.to_csv(quality_path, index=False)

rationale_text = f"""# AI-related state policy activity — screening rationale\n\n## Scope\nThis collection covers all 28 Indian states: {', '.join(STATES)}. It is not a comprehensive census of every AI-policy activity; source and feed coverage varies by state.\n\n## Inclusion rule\nA record is exported as an **Include candidate — verify manually** only when the headline/feed summary contains (1) an AI signal, (2) a concrete activity/action signal, and (3) ev that a government/public-sector actor performed the action or the state itself is the grammatical actor. Activity categories include policy and strategy; institutional initiatives; government AI deployment; partnerships and MoUs; funding and budget; regulation and ethics; and AI infrastructure.\n\n## Exclusions and review\nCommercial AI product news without a state-government role and generic digitisation with no explicit AI connection are not included as candidates. Partial matches are retained in the review queue. All candidates are marked **Pending human verification** because feed metadata alone cannot establish the full context or implementation status.\n\n## ev and limitations\nThe screen uses editable keyword ps over RSS headlines and short feed summaries. It does not read full article bodies or use an LLM. State assignment comes from a state-specific feed/query; an explicit state or city mention is a supporting signal, not a substitute for checking the article. Dates may be absent. Google News RSS results and publisher feeds are incomplete and can change. Failed feeds are listed in `feed_execution_log.csv`.\n\n## Run summary\n- RSS records collected: {len(raw_df)}\n- Deduplicated records screened: {len(screened_df)}\n- AI-related records in review queue: {len(review_df)}\n- Likely policy candidates awaiting human verification: {len(final_df)}\n- Lookback window: {DAYS_BACK} days\n\n## Manual verification required\nOpen each source URL and confirm the state government actor, the specific AI activity, the announcement date, and whether the item is proposed, approved, funded, signed or operational. Correct `activity_type` and `policy_stage` where needed before final submission.\n"""
rationale_path.write_text(rationale_text, encoding="utf-8")

print("Created files:")
for path in [csv_path, xlsx_path, review_path, all_screened_path, raw_path, feed_log_path, quality_path, rationale_path]:
    print(f"- {path} ({path.stat().st_size:,} bytes)")
print(f"\nCandidate records exported: {len(final_df):,}")
print(f"AI-related records in review queue: {len(review_df):,}")
print(f"Output directory: {OUTPUT_DIR}")

# Colab download is optional; local environments simply retain the files.
try:
    from google.colab import files
    files.download(str(csv_path))
    files.download(str(xlsx_path))
except Exception:
    print("Not running in Google Colab; exports are available in the output directory above.")

## 9. Optional: page metadata enrichment (OFF by default)

Use only for a small number of candidate records where the RSS title or summary is insufficient. This fetches metadata (`og:title`, meta description and canonical link) only; it does not extract full article text or bypass paywalls. Respect each publisher's terms and robots policies.

In [ ]:
ENRICH_ARTICLE_PAGES = False
ENRICH_LIMIT = 10


def fetch_page_metadata(url: str) -> dict:
    try:
        response = SESSION.get(url, timeout=REQUEST_TIMEOUT)
        response.raise_for_status()
        soup = BeautifulSoup(response.text, "html.parser")
        def meta_value(**attrs):
            tag = soup.find("meta", attrs=attrs)
            return tag.get("content", "").strip() if tag else ""
        og_title = meta_value(property="og:title") or (soup.title.get_text(" ", strip=True) if soup.title else "")
        description = meta_value(name="description") or meta_value(property="og:description")
        canonical_tag = soup.find("link", rel="canonical")
        canonical_url = canonical_tag.get("href", "").strip() if canonical_tag else ""
        return {"page_title": strip_html(og_title), "page_description": strip_html(description), "canonical_url": normalize_url(canonical_url)}
    except Exception as exc:
        return {"page_title": "", "page_description": "", "canonical_url": "", "page_fetch_error": str(exc)[:200]}


if ENRICH_ARTICLE_PAGES:
    sample = final_df.head(ENRICH_LIMIT).copy()
    metadata_rows = []
    for url in tqdm(sample["source_url"].tolist(), desc="Fetching limited page metadata"):
        metadata_rows.append(fetch_page_metadata(url))
        time.sleep(1.0)
    metadata_df = pd.DataFrame(metadata_rows)
    display(pd.concat([sample.reset_index(drop=True), metadata_df], axis=1))
else:
    print("Page metadata enrichment is disabled. Set ENRICH_ARTICLE_PAGES = True to run it for a small sample.")

## 10. Before submission: human verification checklist

- [ ] Open every candidate article URL. Confirm the reported action is by or on behalf of the assigned state government.
- [ ] Confirm the AI connection is explicit; generic digitisation or a private AI product is not enough.
- [ ] Confirm the action fits one of the defined activity types. Correct the primary type if more than one is present.
- [ ] Verify the policy stage from the article. Do not treat a proposal, MoU or announcement as an implemented service.
- [ ] Confirm the article date and state geography. Do not invent missing dates or infer a state solely from a publisher label when the article points elsewhere.
- [ ] Change `manual_review_status` only after a human has checked the article. Keep a short note of any corrections in the final submission copy.
- [ ] Inspect `ai_policy_review_queue.csv` for partial matches. A relevant record can be missed by a keyword rule; manually add a justified record if you find one.
- [ ] Inspect failed feeds in `feed_execution_log.csv`. A failed feed is not evidence that a state had no AI-policy activity.
- [ ] Report that the collection queries all 28 configured states, but source coverage and collection volume are not uniform.

### Known limitations

1. RSS feeds expose only a subset of publisher coverage and summaries may be truncated.
2. Google News RSS is a discovery endpoint, not a documented API; results and links can vary.
3. The keyword screen can miss synonyms and can produce false positives. It prioritises reproducibility and explainability over perfect recall.
4. State labels are initially assigned from state-specific feeds/queries. Multi-state, national and city-only stories need manual geographic verification.
5. A story reporting an announcement is not proof that an initiative was funded, completed or is operational.
6. The dataset is a news-based monitoring sample, not an authoritative inventory of every state policy.
7. Fuzzy deduplication can merge distinct stories or miss duplicates. Inspect records with `duplicate_count > 1`.

### Suggested next iteration

After manual validation, tune the query groups and screening vocabulary using examples of false positives and missed relevant stories. Add further states with equivalent source research and document the expanded coverage; do not compare states as if collection coverage were uniform.